# Phase 1 — probe at the top vs DPO

Attach the Bayesian reward probe to the **post-final-RMSNorm hidden state** (the exact tensor the
unembedding consumes) and compare it head-to-head with **DPO** on identical preference pairs and
identical LoRA coverage. Testbed: the A/B wrongness task (train the model to prefer *wrong*
answers to comparison questions it gets ≥99% right — deterministic oracle, no capability confound).

**Isotropy argument** (why they should match): for near-spherical unembedding rows, averaging
DPO's per-pair ΔW = W_yw − W_yl leaves only the utility direction ⇒ a single linear head at the
unembedding's input induces ≈ DPO's gradient flow. Run `python isotropy_check.py` for our backbone.

**Pre-registered prediction:** high but **sub-1.0** gradient alignment — DPO's gradient carries a
softmax-normalization component (an implicit imitation pull on the chosen completion) that the
pairwise margin lacks; behaviorally the unanchored probe should displace more than DPO; adding the
DPOP hinge (`anchor>0` in `margin_step`) should raise the cosine.

Phase 2 = same comparison with `ATTACH='block'` and lower `L` (the elbow hypothesis).


In [ ]:
import numpy as np, torch, random, matplotlib.pyplot as plt
from helpers import *

CFG = dict(
    model      = 'Qwen/Qwen2.5-7B',
    attach     = 'final',      # 'final': post-norm read (phase 1) | 'block' (phase 2)
    L          = None,         # None → last layer; set lower for phase 2
    seed       = 0,
    n_train    = 1000, n_eval = 300, n_transfer = 150, n_know_train = 30,
    steps      = 800, batch = 6, lr_probe = 1e-4, lr_dpo = 5e-5, lora_r = 8,
    eval_every = 25, gradcos_batches = 8,
)


In [ ]:
ctx = load_model(CFG['model'])
L = CFG['L'] if CFG['L'] is not None else ctx.n_layers - 1
d = build_data(seed=CFG['seed'], n_train=CFG['n_train'], n_eval=CFG['n_eval'],
               n_transfer=CFG['n_transfer'], n_know_train=CFG['n_know_train'], tok=ctx.tok)
print(f'{ctx.n_layers} blocks | head at L={L} ({CFG["attach"]}) | {len(d.train_pairs)} train pairs')
d.train_pairs[0]['prompt'][-120:], d.train_pairs[0]['wrong'], d.train_pairs[0]['right']


In [ ]:
# completion-end features (disk-cached) + per-layer probes — the decodability curve
key = f"{CFG['model'].replace('/','_')}_s{CFG['seed']}_{len(d.train_pairs)}_{CFG['attach']}"
Xw_tr, Xr_tr = cache_pairend(ctx, d.train_pairs, attach=CFG['attach'], cache_file=f'.f_tr_{key}.npz')
Xw_te, Xr_te = cache_pairend(ctx, d.eval_pairs,  attach=CFG['attach'], cache_file=f'.f_te_{key}.npz')
layer_acc, layer_elbo, heads = fit_probes(ctx, d, Xw_tr, Xr_tr, Xw_te, Xr_te,
                                          cache_file=f'.probes_{key}.pt')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(layer_acc, 'o-'); ax[0].axhline(.5, ls=':', c='gray'); ax[0].set_title('probe acc vs layer')
ax[1].plot(layer_elbo, 's-'); ax[1].set_title('ELBO (evidence proxy)'); plt.show()


## Train the two arms
Same pairs, same LoRA (all blocks), 800 steps, **no early stop** (the over-optimization tail is
data). At every checkpoint: targeted flips, off-menu rate, the frozen head re-read through the
policy, DPO implicit reward, on-menu mass drift, and the **per-block gradient cosine** between the
two losses on fixed shared batches.


In [ ]:
policy = add_lora(ctx, r=CFG['lora_r'])
fh = RewardHead(ctx, heads, L, attach=CFG['attach'])
GC_BATCHES = [random.Random(777).sample(d.train_pairs, CFG['batch']) for _ in range(CFG['gradcos_batches'])]

def run_arm(mode, lr, anchor=0.0, seed=1):
    params = reset_lora(ctx, seed=CFG['seed'] + seed)
    opt = torch.optim.AdamW(params, lr=lr)
    rng = random.Random(4242); ctx.policy.train(); curve = []
    def checkpoint(step):
        gh = goodhart_state(ctx, d, fh); gh['step'] = step
        if mode == 'probe': gh['gradcos'] = grad_cos_vs_dpo(ctx, fh, GC_BATCHES)
        curve.append(gh); ctx.policy.train()
        print(f"  {step:4d}: ab_flip {gh['ab_flip']:.2f} | free flip {gh['free_flip']:.2f} "
              f"offmenu {gh['free_offmenu']:.2f} | head {gh['head_endorse']:.2f} "
              f"dpoR {gh['dpo_margin']:+.1f} Δlp(cho/rej) {gh['dlp_chosen']:+.1f}/{gh['dlp_rejected']:+.1f}", flush=True)
    checkpoint(0)
    for step in range(CFG['steps']):
        batch = rng.sample(d.train_pairs, CFG['batch'])
        opt.zero_grad()
        if mode == 'probe': margin_step(ctx, batch, fh, anchor=anchor)
        else:               dpo_step(ctx, batch)
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
        if (step + 1) % CFG['eval_every'] == 0: checkpoint(step + 1)
    return dict(mode=mode, curve=curve, final=eval_all(ctx, d), rollouts=rollouts(ctx, d))


In [ ]:
res_probe = run_arm('probe', CFG['lr_probe'])


In [ ]:
res_dpo = run_arm('dpo', CFG['lr_dpo'])


In [ ]:
# transfer matrix
keys = [k for k in res_probe['final'] if not k.endswith('_flip')]
print(f"{'eval':>20} {'probe@final':>12} {'dpo':>8}")
for k in keys:
    print(f"{k:>20} {res_probe['final'][k]:>12.2f} {res_dpo['final'][k]:>8.2f}")
print(); print(*res_probe['rollouts'][:6], sep='\n')


In [ ]:
# Goodhart panels + the gradient-alignment measurement
fig, axes = plt.subplots(1, 4, figsize=(21, 4))
for res, c in ((res_probe, 'tab:blue'), (res_dpo, 'tab:orange')):
    cv = res['curve']; st = [x['step'] for x in cv]
    axes[0].plot(st, [x['ab_flip'] for x in cv], 'o-', c=c, label=f"{res['mode']} A/B flip")
    axes[0].plot(st, [x['free_flip'] for x in cv], 's--', c=c, alpha=.6, label=f"{res['mode']} free flip")
    axes[1].plot(st, [x['free_offmenu'] for x in cv], 'o-', c=c, label=res['mode'])
    axes[2].plot(st, [x['head_endorse'] for x in cv], 'o-', c=c, label=res['mode'])
    axes[3].plot(st, [x['dlp_chosen'] for x in cv], 'o-', c=c, label=f"{res['mode']} Δlp chosen")
    axes[3].plot(st, [x['dlp_rejected'] for x in cv], 's--', c=c, alpha=.6, label=f"{res['mode']} Δlp rejected")
for ax, t in zip(axes, ['ORACLE: targeted flips', 'free-form OFF-MENU', 'PROXY: head endorses wrong',
                        'Δ seq-logp vs base (nats), PER SIDE']):
    ax.set_title(t); ax.set_xlabel('step'); ax.legend(fontsize=7)
plt.show()

gcs = [(x['step'], x['gradcos']) for x in res_probe['curve'] if 'gradcos' in x]
fig, ax = plt.subplots(figsize=(7, 4)); cm = plt.get_cmap('viridis')
for i, (st, gc) in enumerate(gcs):
    ax.plot(list(gc.keys()), list(gc.values()), 'o-', c=cm(i / max(len(gcs) - 1, 1)), label=f'step {st}')
ax.axhline(0, ls=':', c='gray'); ax.set_ylim(-1.05, 1.05)
ax.set_xlabel('block'); ax.set_ylabel('cos(∇probe-margin, ∇DPO)'); ax.legend(fontsize=6); plt.show()


## Later: UF-relevant extras (kept in `helpers.py`)
- `sampled_rl_step(ctx, batch, fh, k, kl, pess, ...)` — on-policy RLOO REINFORCE scored by the
  head, KL-in-reward, posterior-uncertainty pessimism (`pess`).
- `fh.filter_round(df, t, min_sigma=...)` — online Bayesian head co-adaptation (t=−1: cooperative,
  fit to the trained preference; t=+1: adversarial truth monitor) with a **variance floor**.
- `margin_step(..., anchor=1.0)` — DPOP hinge on the chosen completion's absolute likelihood.
- `build_data(..., neg_frac=0.3)` — off-menu adversarial negatives for the head.
